# v1


In [ ]:

import rclpy
from rclpy.node import Node
from sensor_msgs.msg import Image
from geometry_msgs.msg import Twist
from cv_bridge import CvBridge
import cv2
import numpy as np

class DualLineFollower(Node):
    def __init__(self):
        super().__init__('dual_line_follower')

        # ===== 可自行調的參數 (HSV上下限) =====
        # 這裡直接寫死在程式中，也可以用 declare_parameter 從外部載入。
        # 黃線
        self.lower_yellow = np.array([20, 100, 100], dtype=np.uint8)
        self.upper_yellow = np.array([30, 255, 255], dtype=np.uint8)
        # 白線
        self.lower_white = np.array([0,   0, 200], dtype=np.uint8)
        self.upper_white = np.array([180, 30, 255], dtype=np.uint8)

        # 其他參數
        self.canny_low = 50
        self.canny_high = 150
        self.hough_threshold = 50
        self.hough_min_line_len = 70
        self.hough_max_line_gap = 200

        # ===== ROS 相關訂閱/發布 =====
        # 訂閱攝影機影像
        self.image_sub = self.create_subscription(
            Image,
            '/camera/image_raw',  # 你在 TurtleBot3 上的實際話題名可能是這個
            self.image_callback,
            10
        )
        # 發布速度指令
        self.cmd_vel_pub = self.create_publisher(Twist, '/cmd_vel', 10)

        # OpenCV Bridge
        self.bridge = CvBridge()

        # 週期性控制 (每 0.1 秒跑一次 control_loop)
        self.timer = self.create_timer(0.1, self.control_loop)

        # 儲存影像處理結果
        self.error = None     # 影像中心偏移 (越正代表線在右邊，越負線在左邊)
        self.frame_width = 640  # 預設影像寬度，等拿到第一張圖再更新

    def image_callback(self, msg):
        # ===== ROS Image => OpenCV Image =====
        cv_image = self.bridge.imgmsg_to_cv2(msg, desired_encoding='bgr8')
        h, w, _ = cv_image.shape
        self.frame_width = w

        # ===== 轉HSV並做遮罩 =====
        hsv = cv2.cvtColor(cv_image, cv2.COLOR_BGR2HSV)
        mask_yellow = cv2.inRange(hsv, self.lower_yellow, self.upper_yellow)
        mask_white = cv2.inRange(hsv, self.lower_white, self.upper_white)

        # ===== 先用黃線，再用白線 =====
        line_yellow = self.find_average_line(mask_yellow, cv_image)
        line_white = self.find_average_line(mask_white, cv_image)

        # 先看黃線
        if line_yellow is not None:
            # line_yellow: (x1, y1, x2, y2)
            x1, y1, x2, y2 = line_yellow
            center_x = (x1 + x2) // 2
            self.error = center_x - (w // 2)  # 偏移量
            # 這裡你可以印一下，或在 log 訊息看到
            # self.get_logger().info(f"Yellow line => error={self.error}")
        # 找不到黃線 => 看白線
        elif line_white is not None:
            x1, y1, x2, y2 = line_white
            center_x = (x1 + x2) // 2
            self.error = center_x - (w // 2)
            # self.get_logger().info(f"White line => error={self.error}")
        else:
            # 兩者皆無 => 代表沒偵測到線 => 之後 control_loop 會給一個搜尋動作
            self.error = None

    def control_loop(self):
        """
        根據 self.error 決定 cmd_vel，並發布出去
        """
        twist = Twist()

        if self.error is not None:
            # 有找到線 => 前進 & 轉彎
            # 線速度(可自行調整)
            twist.linear.x = 0.2

            # 角速度(簡單P控制 => angular = -Kp * error)
            # error 是「線中點 - 影像中心」，error>0 表示線在右側，需往右轉(負角速度)
            # 所以這裡要加個負號
            Kp = 0.002  # 比例係數，視情況可調大或調小
            twist.angular.z = -Kp * self.error
        else:
            # 沒看到黃線也沒看到白線 => 原地旋轉找路
            twist.linear.x = 0.0
            twist.angular.z = 0.3

        # 發布
        self.cmd_vel_pub.publish(twist)

    def find_average_line(self, mask, origin_bgr):
        """
        用 Canny + Hough 找出線段，並做簡易「平均線」回傳 (x1, y1, x2, y2).
        若沒有線則回傳 None
        """
        # Canny邊緣
        edges = cv2.Canny(mask, self.canny_low, self.canny_high)

        # Hough
        lines = cv2.HoughLinesP(
            edges,
            1, np.pi/180,
            self.hough_threshold,
            minLineLength=self.hough_min_line_len,
            maxLineGap=self.hough_max_line_gap
        )
        if lines is None or len(lines) == 0:
            return None

        # 簡易平均 => 取出所有 (x,y) 進行線性回歸
        x_coords = []
        y_coords = []
        for line in lines:
            x1, y1, x2, y2 = line[0]
            x_coords.extend([x1, x2])
            y_coords.extend([y1, y2])

        # 做一個 y->x 的 一次多項式擬合
        # 最後在畫面最底 (y=h) 和某個上方 (y=0.6h) 去求 x
        h, w, _ = origin_bgr.shape
        try:
            poly = np.polyfit(y_coords, x_coords, 1)
        except:
            return None

        y1, y2 = h, int(h * 0.6)
        x1 = int(np.polyval(poly, y1))
        x2 = int(np.polyval(poly, y2))

        return (x1, y1, x2, y2)

def main(args=None):
    rclpy.init(args=args)
    node = DualLineFollower()
    rclpy.spin(node)
    node.destroy_node()
    rclpy.shutdown()

if __name__ == '__main__':
    main()


In [ ]:
#!/usr/bin/env python3

import rclpy
from rclpy.node import Node
from sensor_msgs.msg import Image
from geometry_msgs.msg import Twist
from cv_bridge import CvBridge
import cv2
import numpy as np
import json
import os

# ---------------------------------------------------
# 分離設定檔：Trackbar & ROI
# ---------------------------------------------------
SETTINGS_FILE = "trackbar_settings_lu.json"  # Trackbar 設定檔
ROI_FILE = "roi_polygon.json"                # 多邊形 ROI 設定檔

DEFAULT_VALUES = {
    # 黃色上下限
    "L Y H": 20,   "U Y H": 30,
    "L Y S": 100,  "U Y S": 255,
    "L Y V": 100,  "U Y V": 255,

    # 白色上下限
    "L W H": 0,    "U W H": 255,
    "L W S": 0,    "U W S": 50,
    "L W V": 180,  "U W V": 255,

    # Canny
    "Canny Low": 50,
    "Canny High": 150,

    # Hough 參數
    "Hough Threshold": 50,
    "Hough MinLength": 70,
    "Hough MaxGap": 200
}


class MyLineDetectorSub(Node):
    def __init__(self):
        super().__init__('my_line_detector_sub')

        # 1) 訂閱相機影像
        self.bridge = CvBridge()
        self.image_sub = self.create_subscription(
            Image,
            '/camera/image_raw',
            self.image_callback,
            10
        )
        # === 原本的全域變數改成屬性 ===
        self.last_frame = None
        self.picking_polygon = False
        self.user_polygon_points = []


        # 建立 Trackbars
        self.create_trackbar()

        # 建立顯示視窗
        cv2.namedWindow("Road Detection", cv2.WINDOW_NORMAL)
        cv2.namedWindow("Edges", cv2.WINDOW_NORMAL)
        cv2.namedWindow("Yellow Mask", cv2.WINDOW_NORMAL)
        cv2.namedWindow("White Mask", cv2.WINDOW_NORMAL)
        cv2.namedWindow("Hough Lines", cv2.WINDOW_NORMAL)

        # 滑鼠 callback => 用於 ROI 多邊形
        cv2.setMouseCallback("Road Detection", self.on_mouse)

    # ------------------- 原本的函式搬進來 -------------------
    def load_settings(self):
        if os.path.exists(SETTINGS_FILE):
            with open(SETTINGS_FILE, "r") as f:
                return json.load(f)
        return DEFAULT_VALUES

    def save_settings(self, values):
        with open(SETTINGS_FILE, "w") as f:
            json.dump(values, f, indent=4)

    def load_roi_polygon(self):
        if os.path.exists(ROI_FILE):
            with open(ROI_FILE, 'r') as f:
                data = json.load(f)
                if "polygon" in data and len(data["polygon"]) >= 3:
                    return data["polygon"]
        return None

    def save_roi_polygon(self, points):
        data = {"polygon": points}
        with open(ROI_FILE, 'w') as f:
            json.dump(data, f, indent=4)

    def clamp(self, val, vmin, vmax):
        return max(vmin, min(val, vmax))

    def pickRange(self, color, frame):
        r = cv2.selectROI("Road Detection", frame, fromCenter=False)
        x, y, w, h = [int(i) for i in r]
        if w == 0 or h == 0:
            return
        roi_bgr = frame[y:y+h, x:x+w]
        roi_hsv = cv2.cvtColor(roi_bgr, cv2.COLOR_BGR2HSV)
        mean_hsv = cv2.mean(roi_hsv)
        h_mean, s_mean, v_mean = mean_hsv[:3]

        h_low  = self.clamp(int(h_mean - 10), 0, 179)
        h_high = self.clamp(int(h_mean + 10), 0, 179)
        s_low  = self.clamp(int(s_mean - 40), 0, 255)
        s_high = self.clamp(int(s_mean + 40), 0, 255)
        v_low  = self.clamp(int(v_mean - 40), 0, 255)
        v_high = self.clamp(int(v_mean + 40), 0, 255)

        if color == 'yellow':
            cv2.setTrackbarPos('L Y H', 'Trackbars', h_low)
            cv2.setTrackbarPos('U Y H', 'Trackbars', h_high)
            cv2.setTrackbarPos('L Y S', 'Trackbars', s_low)
            cv2.setTrackbarPos('U Y S', 'Trackbars', s_high)
            cv2.setTrackbarPos('L Y V', 'Trackbars', v_low)
            cv2.setTrackbarPos('U Y V', 'Trackbars', v_high)
        else:  # white
            cv2.setTrackbarPos('L W H', 'Trackbars', 0)
            cv2.setTrackbarPos('U W H', 'Trackbars', 179)
            cv2.setTrackbarPos('L W S', 'Trackbars', s_low)
            cv2.setTrackbarPos('U W S', 'Trackbars', s_high)
            cv2.setTrackbarPos('L W V', 'Trackbars', v_low)
            cv2.setTrackbarPos('U W V', 'Trackbars', v_high)

    def nothing(self, x):
        pass

    def get_trackbar_values(self):
        current = self.load_settings()
        values = {}
        for key in current.keys():
            if key in DEFAULT_VALUES:
                try:
                    v = cv2.getTrackbarPos(key, "Trackbars")
                    values[key] = v
                except:
                    values[key] = current[key]
            else:
                values[key] = current[key]
        return values

    def create_trackbar(self):
        cv2.namedWindow("Trackbars", cv2.WINDOW_NORMAL)
        cv2.resizeWindow("Trackbars", 400, 600)

        settings = self.load_settings()

        trackbar_pairs = [
            ("Y H (L)", "L Y H", (0, 179)),
            ("Y H (U)", "U Y H", (0, 179)),
            ("Y S (L)", "L Y S", (0, 255)),
            ("Y S (U)", "U Y S", (0, 255)),
            ("Y V (L)", "L Y V", (0, 255)),
            ("Y V (U)", "U Y V", (0, 255)),

            ("W H (L)", "L W H", (0, 179)),
            ("W H (U)", "U W H", (0, 179)),
            ("W S (L)", "L W S", (0, 255)),
            ("W S (U)", "U W S", (0, 255)),
            ("W V (L)", "L W V", (0, 255)),
            ("W V (U)", "U W V", (0, 255)),

            ("Canny Low",  "Canny Low",  (0, 255)),
            ("Canny High", "Canny High", (0, 255)),

            ("Hough Thresh",   "Hough Threshold", (1, 200)),
            ("Hough MinLen",   "Hough MinLength", (1, 300)),
            ("Hough MaxGap",   "Hough MaxGap",    (1, 300)),
        ]

        for display_name, key, (min_val, max_val) in trackbar_pairs:
            initial = settings.get(key, DEFAULT_VALUES[key])
            cv2.createTrackbar(key, "Trackbars", initial, max_val, self.nothing)

    # ---------------------------------------------------
    # 多邊形 ROI
    # ---------------------------------------------------
    def on_mouse(self, event, x, y, flags, param):
        if self.picking_polygon and event == cv2.EVENT_LBUTTONDOWN:
            self.user_polygon_points.append((x, y))

    def pick_polygon_interactive(self):
        self.picking_polygon = True
        self.user_polygon_points = []
        print("[INFO] 開始指定多邊形 ROI，請在 Road Detection 視窗中左鍵點選多個頂點。")
        print("[INFO] 按 Enter 或 Esc 結束，按 Backspace 刪除最後一個點。")

    def draw_polygon_preview(self, frame):
        for i, pt in enumerate(self.user_polygon_points):
            cv2.circle(frame, pt, 5, (0, 0, 255), -1)
            if i > 0:
                cv2.line(frame, self.user_polygon_points[i-1], pt, (0, 0, 255), 2)
        if len(self.user_polygon_points) > 1:
            cv2.line(frame, self.user_polygon_points[-1], self.user_polygon_points[0], (0, 0, 255), 1)

    # ---------------------------------------------------
    # 核心影像處理
    # ---------------------------------------------------
    def preprocess_image(self, image, lower_yellow, upper_yellow, lower_white, upper_white, canny_low, canny_high):
        hsv = cv2.cvtColor(image, cv2.COLOR_BGR2HSV)
        yellow_mask = cv2.inRange(hsv, lower_yellow, upper_yellow)
        white_mask = cv2.inRange(hsv, lower_white, upper_white)
        mask = cv2.bitwise_or(yellow_mask, white_mask)
        edges = cv2.Canny(mask, canny_low, canny_high)
        return edges, yellow_mask, white_mask

    def region_of_interest(self, image):
        custom_polygon = self.load_roi_polygon()
        if custom_polygon is not None:
            pts = np.array([custom_polygon], dtype=np.int32)
        else:
            height, width = image.shape[:2]
            pts = np.array([[
                (0, height), (width, height),
                (int(width * 0.9), int(height * 0.5)),
                (int(width * 0.1), int(height * 0.5))
            ]], dtype=np.int32)
        mask = np.zeros_like(image)
        cv2.fillPoly(mask, pts, 255)
        return cv2.bitwise_and(image, mask)

    def detect_lines(self, image):
        values = self.get_trackbar_values()
        h_thresh = values["Hough Threshold"]
        h_minlen = values["Hough MinLength"]
        h_maxgap = values["Hough MaxGap"]
        lines = cv2.HoughLinesP(
            image, 1, np.pi / 180,
            h_thresh, minLineLength=h_minlen, maxLineGap=h_maxgap
        )
        return lines

    def average_line(self, image, lines):
        if lines is None or len(lines) == 0:
            return None
        x_coords, y_coords = [], []
        for line in lines:
            x1, y1, x2, y2 = line[0]
            x_coords.extend([x1, x2])
            y_coords.extend([y1, y2])
        poly = np.polyfit(y_coords, x_coords, 1)
        height, width = image.shape[:2]
        y1, y2 = height, int(height * 0.6)
        x1, x2 = int(np.polyval(poly, y1)), int(np.polyval(poly, y2))
        return x1, y1, x2, y2

    def draw_lines(self, image, line):
        line_image = np.zeros_like(image)
        if line is not None:
            cv2.line(line_image, (line[0], line[1]), (line[2], line[3]), (0, 255, 0), 5)
        return line_image

    def draw_center_line(self, image, yellow_line, white_line):
        if yellow_line is not None and white_line is not None:
            center_x1 = (yellow_line[0] + white_line[0]) // 2
            center_x2 = (yellow_line[2] + white_line[2]) // 2
        elif yellow_line is not None:
            center_x1 = yellow_line[0] + 300
            center_x2 = yellow_line[2] + 300
        elif white_line is not None:
            center_x1 = white_line[0] - 300
            center_x2 = white_line[2] - 300
        else:
            return
        height, width = image.shape[:2]
        y1, y2 = height, int(height * 0.6)
        cv2.line(image, (center_x1, y1), (center_x2, y2), (0, 0, 255), 3)

    # ------------------- 主流程: run() -------------------
    def run(self):
        while self.cap.isOpened():
            ret, frame = self.cap.read()
            if not ret:
                break

            self.last_frame = frame.copy()

            # 如果正在 pick polygon，就畫出暫存頂點
            if self.picking_polygon:
                self.draw_polygon_preview(frame)

            values = self.get_trackbar_values()
            lower_yellow = np.array([values["L Y H"], values["L Y S"], values["L Y V"]])
            upper_yellow = np.array([values["U Y H"], values["U Y S"], values["U Y V"]])
            lower_white = np.array([values["L W H"], values["L W S"], values["L W V"]])
            upper_white = np.array([values["U W H"], values["U W S"], values["U W V"]])
            canny_low, canny_high = values["Canny Low"], values["Canny High"]

            edges, yellow_mask, white_mask = self.preprocess_image(
                frame, lower_yellow, upper_yellow, lower_white, upper_white,
                canny_low, canny_high
            )

            cv2.imshow("Edges", edges)
            cv2.imshow("Yellow Mask", yellow_mask)
            cv2.imshow("White Mask", white_mask)

            roi_edges = self.region_of_interest(edges)
            # 對yellow/white_mask也做 ROI
            roi_yellow = self.region_of_interest(yellow_mask)
            roi_white = self.region_of_interest(white_mask)

            y_lines = self.detect_lines(roi_yellow)
            w_lines = self.detect_lines(roi_white)
            yellow_line = self.average_line(frame, y_lines)
            white_line = self.average_line(frame, w_lines)

            # Hough Lines
            lines_image = np.zeros_like(frame)
            if y_lines is not None:
                for line in y_lines:
                    x1, y1, x2, y2 = line[0]
                    cv2.line(lines_image, (x1,y1), (x2,y2), (0, 255, 255), 2)
            if w_lines is not None:
                for line in w_lines:
                    x1, y1, x2, y2 = line[0]
                    cv2.line(lines_image, (x1,y1), (x2,y2), (255, 255, 255), 2)
            cv2.imshow("Hough Lines", lines_image)

            line_image = self.draw_lines(frame, yellow_line)
            line_image += self.draw_lines(frame, white_line)
            combined = cv2.addWeighted(frame, 0.8, line_image, 1, 1)
            self.draw_center_line(combined, yellow_line, white_line)

            cv2.imshow("Road Detection", combined)

            key = cv2.waitKey(1) & 0xFF
            if key == ord('q'):
                break
            elif key == ord('y'):
                self.pickRange('yellow', self.last_frame)
            elif key == ord('w'):
                self.pickRange('white', self.last_frame)
            elif key == ord('i'):
                self.pick_polygon_interactive()
            elif key == 8:  # Backspace => 刪除最後一個點
                if self.picking_polygon and self.user_polygon_points:
                    self.user_polygon_points.pop()
            elif key in [13, 27]:  # Enter 或 Esc => 結束多邊形
                if self.picking_polygon and len(self.user_polygon_points) >= 3:
                    self.save_roi_polygon(self.user_polygon_points)
                    print(f"[INFO] 多邊形 ROI 已存檔 => {self.user_polygon_points}")
                self.picking_polygon = False

        # 離開前存一下滑桿設定
        updated_values = self.get_trackbar_values()
        self.save_settings(updated_values)

        self.cap.release()
        cv2.destroyAllWindows()


# ------------------- 主程式入口 -------------------
def main():
    detector = MyLineDetector(video_source='testv5.mp4')
    detector.run()

if __name__ == "__main__":
    main()



# control q huiyouwent


In [ ]:
#!/usr/bin/env python3

import rclpy
from rclpy.node import Node
from sensor_msgs.msg import Image
from geometry_msgs.msg import Twist  # 如果你想發 /cmd_vel，可加這個
from cv_bridge import CvBridge

import cv2
import numpy as np
import json
import os

SETTINGS_FILE = "trackbar_settings_lu.json"
ROI_FILE = "roi_polygon.json"

DEFAULT_VALUES = {
    # 黃色上下限
    "L Y H": 20,   "U Y H": 30,
    "L Y S": 100,  "U Y S": 255,
    "L Y V": 100,  "U Y V": 255,

    # 白色上下限
    "L W H": 0,    "U W H": 255,
    "L W S": 0,    "U W S": 50,
    "L W V": 180,  "U W V": 255,

    # Canny
    "Canny Low": 50,
    "Canny High": 150,

    # Hough
    "Hough Threshold": 50,
    "Hough MinLength": 70,
    "Hough MaxGap": 200
}

class MyLineDetectorSub(Node):
    def __init__(self):
        super().__init__('my_line_detector_sub')

        # 1) 訂閱相機影像
        self.bridge = CvBridge()
        self.image_sub = self.create_subscription(
            Image,
            '/camera/image_raw',
            self.image_callback,
            10
        )

        # 2) 發布速度 (如果想控制機器人，就開這個)
        # self.cmd_vel_pub = self.create_publisher(Twist, '/cmd_vel', 10)

        # 3) GUI 相關狀態
        self.last_frame = None    # 最新影像
        self.new_frame_available = False

        self.picking_polygon = False
        self.user_polygon_points = []

        # 建立 Trackbars
        self.create_trackbar()

        # 建立 OpenCV 視窗
        cv2.namedWindow("Road Detection", cv2.WINDOW_NORMAL)
        cv2.namedWindow("Edges", cv2.WINDOW_NORMAL)
        cv2.namedWindow("Yellow Mask", cv2.WINDOW_NORMAL)
        cv2.namedWindow("White Mask", cv2.WINDOW_NORMAL)
        cv2.namedWindow("Hough Lines", cv2.WINDOW_NORMAL)

        # 設定滑鼠callback => ROI 多邊形
        cv2.setMouseCallback("Road Detection", self.on_mouse)

        # 4) 建立定時器，每 0.05秒 (20Hz) 做一次 process_frame
        self.timer = self.create_timer(0.05, self.process_frame)

        # 讀取 (或初始化) Trackbar 設定
        self.get_logger().info("MyLineDetectorSub Node Started. Subscribe /camera/image_raw")

    # ================ 影像回呼 =================
    def image_callback(self, msg):
        # ROS Image => OpenCV image
        frame = self.bridge.imgmsg_to_cv2(msg, desired_encoding='bgr8')
        self.last_frame = frame
        self.new_frame_available = True

    # ================ 定時器: 處理最新影像+GUI =================
    def process_frame(self):
        if not self.new_frame_available or self.last_frame is None:
            return

        # 取出最新影像
        frame = self.last_frame.copy()
        self.new_frame_available = False  # 表示已處理

        # 如果正在 pick polygon，就畫出暫存頂點
        if self.picking_polygon:
            self.draw_polygon_preview(frame)

        # 讀取 Trackbar 參數
        values = self.get_trackbar_values()
        lower_yellow = np.array([values["L Y H"], values["L Y S"], values["L Y V"]])
        upper_yellow = np.array([values["U Y H"], values["U Y S"], values["U Y V"]])
        lower_white = np.array([values["L W H"], values["L W S"], values["L W V"]])
        upper_white = np.array([values["U W H"], values["U W S"], values["U W V"]])
        canny_low, canny_high = values["Canny Low"], values["Canny High"]

        # 做預處理
        edges, yellow_mask, white_mask = self.preprocess_image(
            frame, lower_yellow, upper_yellow,
            lower_white, upper_white,
            canny_low, canny_high
        )

        cv2.imshow("Edges", edges)
        cv2.imshow("Yellow Mask", yellow_mask)
        cv2.imshow("White Mask", white_mask)

        # 做 ROI
        roi_edges = self.region_of_interest(edges)
        roi_yellow = self.region_of_interest(yellow_mask)
        roi_white = self.region_of_interest(white_mask)

        # 偵測線
        y_lines = self.detect_lines(roi_yellow)
        w_lines = self.detect_lines(roi_white)
        yellow_line = self.average_line(frame, y_lines)
        white_line = self.average_line(frame, w_lines)

        # 繪製線段
        lines_image = np.zeros_like(frame)
        if y_lines is not None:
            for line in y_lines:
                x1, y1, x2, y2 = line[0]
                cv2.line(lines_image, (x1,y1), (x2,y2), (0, 255, 255), 2)
        if w_lines is not None:
            for line in w_lines:
                x1, y1, x2, y2 = line[0]
                cv2.line(lines_image, (x1,y1), (x2,y2), (255, 255, 255), 2)
        cv2.imshow("Hough Lines", lines_image)

        # 畫平均線
        line_image = self.draw_lines(frame, yellow_line)
        line_image += self.draw_lines(frame, white_line)
        combined = cv2.addWeighted(frame, 0.8, line_image, 1, 1)
        self.draw_center_line(combined, yellow_line, white_line)

        cv2.imshow("Road Detection", combined)

        # 如果想控制機器人 => 這邊可以發布 /cmd_vel
        # e.g. self.cmd_vel_pub.publish(twist)

        # 處理鍵盤事件
        key = cv2.waitKey(1) & 0xFF
        if key == ord('q'):
            self.destroy_node()  # 讓spin結束
            rclpy.shutdown()
            return
        elif key == ord('i'):
            self.pick_polygon_interactive()
        elif key == 8:  # Backspace => 刪除最後一個點
            if self.picking_polygon and self.user_polygon_points:
                self.user_polygon_points.pop()
        elif key in [13, 27]:  # Enter 或 Esc => 結束多邊形
            if self.picking_polygon and len(self.user_polygon_points) >= 3:
                self.save_roi_polygon(self.user_polygon_points)
                self.get_logger().info(f"ROI saved => {self.user_polygon_points}")
            self.picking_polygon = False

    # ================ 以下保留你原本的函式 =================

    def load_settings(self):
        if os.path.exists(SETTINGS_FILE):
            with open(SETTINGS_FILE, "r") as f:
                return json.load(f)
        return DEFAULT_VALUES

    def save_settings(self, values):
        with open(SETTINGS_FILE, "w") as f:
            json.dump(values, f, indent=4)

    def load_roi_polygon(self):
        if os.path.exists(ROI_FILE):
            with open(ROI_FILE, 'r') as f:
                data = json.load(f)
                if "polygon" in data and len(data["polygon"]) >= 3:
                    return data["polygon"]
        return None

    def save_roi_polygon(self, points):
        data = {"polygon": points}
        with open(ROI_FILE, 'w') as f:
            json.dump(data, f, indent=4)

    def nothing(self, x):
        pass

    def get_trackbar_values(self):
        current = self.load_settings()
        values = {}
        for key in current.keys():
            if key in DEFAULT_VALUES:
                try:
                    v = cv2.getTrackbarPos(key, "Trackbars")
                    values[key] = v
                except:
                    values[key] = current[key]
            else:
                values[key] = current[key]
        return values

    def create_trackbar(self):
        cv2.namedWindow("Trackbars", cv2.WINDOW_NORMAL)
        cv2.resizeWindow("Trackbars", 400, 600)
        settings = self.load_settings()
        trackbar_pairs = [
            ("Y H (L)", "L Y H", (0, 179)),
            ("Y H (U)", "U Y H", (0, 179)),
            ("Y S (L)", "L Y S", (0, 255)),
            ("Y S (U)", "U Y S", (0, 255)),
            ("Y V (L)", "L Y V", (0, 255)),
            ("Y V (U)", "U Y V", (0, 255)),

            ("W H (L)", "L W H", (0, 179)),
            ("W H (U)", "U W H", (0, 179)),
            ("W S (L)", "L W S", (0, 255)),
            ("W S (U)", "U W S", (0, 255)),
            ("W V (L)", "L W V", (0, 255)),
            ("W V (U)", "U W V", (0, 255)),

            ("Canny Low",  "Canny Low",  (0, 255)),
            ("Canny High", "Canny High", (0, 255)),

            ("Hough Thresh",   "Hough Threshold", (1, 200)),
            ("Hough MinLen",   "Hough MinLength", (1, 300)),
            ("Hough MaxGap",   "Hough MaxGap",    (1, 300)),
        ]
        for display_name, key, (min_val, max_val) in trackbar_pairs:
            initial = settings.get(key, DEFAULT_VALUES[key])
            cv2.createTrackbar(key, "Trackbars", initial, max_val, self.nothing)

    # ROI 多邊形
    def on_mouse(self, event, x, y, flags, param):
        if self.picking_polygon and event == cv2.EVENT_LBUTTONDOWN:
            self.user_polygon_points.append((x, y))

    def pick_polygon_interactive(self):
        self.picking_polygon = True
        self.user_polygon_points = []
        print("[INFO] pick polygon: left-click multiple points in Road Detection window.")
        print("[INFO] press Enter or Esc to finish, Backspace to undo last point.")

    def draw_polygon_preview(self, frame):
        for i, pt in enumerate(self.user_polygon_points):
            cv2.circle(frame, pt, 5, (0, 0, 255), -1)
            if i > 0:
                cv2.line(frame, self.user_polygon_points[i-1], pt, (0, 0, 255), 2)
        if len(self.user_polygon_points) > 1:
            cv2.line(frame, self.user_polygon_points[-1], self.user_polygon_points[0], (0, 0, 255), 1)

    def preprocess_image(self, image, lower_yellow, upper_yellow,
                         lower_white, upper_white, canny_low, canny_high):
        hsv = cv2.cvtColor(image, cv2.COLOR_BGR2HSV)
        yellow_mask = cv2.inRange(hsv, lower_yellow, upper_yellow)
        white_mask = cv2.inRange(hsv, lower_white, upper_white)
        mask = cv2.bitwise_or(yellow_mask, white_mask)
        edges = cv2.Canny(mask, canny_low, canny_high)
        return edges, yellow_mask, white_mask

    def region_of_interest(self, image):
        poly = self.load_roi_polygon()
        if poly is not None and len(poly) >= 3:
            pts = np.array([poly], dtype=np.int32)
        else:
            height, width = image.shape[:2]
            pts = np.array([[
                (0, height), (width, height),
                (int(width*0.9), int(height*0.5)),
                (int(width*0.1), int(height*0.5))
            ]], dtype=np.int32)
        mask = np.zeros_like(image)
        cv2.fillPoly(mask, pts, 255)
        return cv2.bitwise_and(image, mask)

    def detect_lines(self, edges):
        tv = self.get_trackbar_values()
        h_thresh = tv["Hough Threshold"]
        h_minlen = tv["Hough MinLength"]
        h_maxgap = tv["Hough MaxGap"]
        lines = cv2.HoughLinesP(
            edges, 1, np.pi/180,
            h_thresh, minLineLength=h_minlen, maxLineGap=h_maxgap
        )
        return lines

    def average_line(self, image, lines):
        if lines is None or len(lines) == 0:
            return None
        x_coords, y_coords = [], []
        for line in lines:
            x1, y1, x2, y2 = line[0]
            x_coords.extend([x1, x2])
            y_coords.extend([y1, y2])
        try:
            poly = np.polyfit(y_coords, x_coords, 1)
        except:
            return None
        h, w = image.shape[:2]
        y1, y2 = h, int(h*0.6)
        x1, x2 = int(np.polyval(poly, y1)), int(np.polyval(poly, y2))
        return (x1, y1, x2, y2)

    def draw_lines(self, image, line):
        line_image = np.zeros_like(image)
        if line is not None:
            x1, y1, x2, y2 = line
            cv2.line(line_image, (x1,y1), (x2,y2), (0,255,0), 5)
        return line_image

    def draw_center_line(self, image, yellow_line, white_line):
        if yellow_line is not None and white_line is not None:
            center_x1 = (yellow_line[0] + white_line[0]) // 2
            center_x2 = (yellow_line[2] + white_line[2]) // 2
        elif yellow_line is not None:
            center_x1 = yellow_line[0] + 300
            center_x2 = yellow_line[2] + 300
        elif white_line is not None:
            center_x1 = white_line[0] - 300
            center_x2 = white_line[2] - 300
        else:
            return
        h, w = image.shape[:2]
        y1, y2 = h, int(h*0.6)
        cv2.line(image, (center_x1,y1), (center_x2,y2), (0,0,255), 3)

def main(args=None):
    rclpy.init(args=args)
    node = MyLineDetectorSub()
    rclpy.spin(node)
    # 結束後
    cv2.destroyAllWindows()
    node.destroy_node()
    rclpy.shutdown()

if __name__ == "__main__":
    main()

# one line land follow

In [ ]:
import rclpy
from rclpy.node import Node
from sensor_msgs.msg import Image
from geometry_msgs.msg import Twist
from cv_bridge import CvBridge

import cv2
import numpy as np

class SingleLineTracker(Node):
    def __init__(self):
        super().__init__('single_line_tracker')
        # 1. 訂閱相機影像
        self.image_sub = self.create_subscription(
            Image, 
            '/camera/image_raw', 
            self.image_callback, 
            10
        )
        # 2. 發布控制速度 (cmd_vel)
        self.cmd_vel_pub = self.create_publisher(Twist, '/cmd_vel', 10)
        self.bridge = CvBridge()
        self.last_frame = None
        self.new_frame_available = False

        # 3. 使用 Timer 定時處理影像 (20 Hz)
        self.timer = self.create_timer(0.05, self.process_frame)

        # 4. HSV 閥值定義
        # 黃色線範圍 (主線)
        self.lower_yellow = np.array([20, 100, 100])
        self.upper_yellow = np.array([30, 255, 255])
        # 白色線範圍 (備用線)
        self.lower_white = np.array([0, 0, 180])
        self.upper_white = np.array([255, 50, 255])
        # 判斷質心的最小面積閾值 (避免雜訊干擾)
        self.area_threshold = 1000

        self.get_logger().info("單線尋路節點已啟動。優先找黃線，找不到則切換白線。")

    def image_callback(self, msg):
        try:
            # ROS Image 消息轉成 OpenCV 格式
            frame = self.bridge.imgmsg_to_cv2(msg, desired_encoding='bgr8')
            self.last_frame = frame
            self.new_frame_available = True
        except Exception as e:
            self.get_logger().error("影像轉換錯誤: " + str(e))

    def process_frame(self):
        if not self.new_frame_available or self.last_frame is None:
            return

        # 取一份當前最新影像
        frame = self.last_frame.copy()
        self.new_frame_available = False

        # 將影像轉換到 HSV 色彩空間
        hsv = cv2.cvtColor(frame, cv2.COLOR_BGR2HSV)

        # --- 優先使用黃色遮罩 ---
        mask_yellow = cv2.inRange(hsv, self.lower_yellow, self.upper_yellow)
        # 顯示黃色遮罩 (debug)
        cv2.imshow("Yellow Mask", mask_yellow)
        center = self.get_line_center(mask_yellow)
        detected_color = "yellow"

        # --- 如果黃色線無法有效偵測，則嘗試白色遮罩 ---
        if center is None:
            mask_white = cv2.inRange(hsv, self.lower_white, self.upper_white)
            cv2.imshow("White Mask", mask_white)
            center = self.get_line_center(mask_white)
            detected_color = "white"
        else:
            # 也顯示白色遮罩方便觀察
            mask_white = cv2.inRange(hsv, self.lower_white, self.upper_white)
            cv2.imshow("White Mask", mask_white)

        # 計算畫面中心 (假設相機正前方)
        h, w, _ = frame.shape
        frame_center = w // 2

        if center is not None:
            cx, cy = center
            # 在畫面上標示出線的中心點
            cv2.circle(frame, (cx, cy), 5, (0, 0, 255), -1)
            # 計算中心誤差
            error = cx - frame_center
            cv2.putText(frame, f"Error: {error}", (10, 30), 
                        cv2.FONT_HERSHEY_SIMPLEX, 1, (0, 255, 0), 2)
            
            # 基本的比例控制 (P control)
            linear_speed = 0.05  # 固定前進速度
            angular_speed = -0.005 * error  # 根據誤差決定轉向 (可根據實際情況調參)
            
            twist = Twist()
            twist.linear.x = linear_speed
            twist.angular.z = angular_speed
            self.cmd_vel_pub.publish(twist)
            self.get_logger().info(
                f"檢測到 {detected_color} 線, 中心在 ({cx},{cy})，error: {error}"
            )
        else:
            # 黃線和白線均無法偵測到，原地旋轉以搜尋
            twist = Twist()
            twist.linear.x = 0.0
            twist.angular.z = 0.3  # 原地右轉 (可調整轉速)
            self.cmd_vel_pub.publish(twist)
            self.get_logger().info("未偵測到任何線，開始旋轉搜尋。")

        # 顯示偵測後的調試畫面
        cv2.imshow("Line Tracking", frame)
        key = cv2.waitKey(1)
        if key == ord('q'):
            self.get_logger().info("使用者請求退出。")
            self.destroy_node()
            rclpy.shutdown()
            cv2.destroyAllWindows()

    def get_line_center(self, mask):
        """
        利用影像 moments 求遮罩內的質心。
        若 m00 面積大於設定閾值，回傳 (cx, cy)；否則回傳 None
        """
        M = cv2.moments(mask)
        if M["m00"] > self.area_threshold:
            cx = int(M["m10"] / M["m00"])
            cy = int(M["m01"] / M["m00"])
            return (cx, cy)
        else:
            return None

def main(args=None):
    rclpy.init(args=args)
    node = SingleLineTracker()
    try:
        rclpy.spin(node)
    except KeyboardInterrupt:
        node.get_logger().info("鍵盤中斷，節點關閉。")
    finally:
        node.destroy_node()
        rclpy.shutdown()
        cv2.destroyAllWindows()

if __name__ == '__main__':
    main()

# single line pianyi


In [ ]:
import rclpy
from rclpy.node import Node
from sensor_msgs.msg import Image
from geometry_msgs.msg import Twist
from cv_bridge import CvBridge

import cv2
import numpy as np

class SingleLineTracker(Node):
    def __init__(self):
        super().__init__('single_line_tracker')
        # 1. 訂閱相機影像
        self.image_sub = self.create_subscription(
            Image, 
            '/camera/image_raw', 
            self.image_callback, 
            10
        )
        # 2. 發布控制速度 (cmd_vel)
        self.cmd_vel_pub = self.create_publisher(Twist, '/cmd_vel', 10)
        self.bridge = CvBridge()
        self.last_frame = None
        self.new_frame_available = False

        # 3. 使用 Timer 定時處理影像 (20 Hz)
        self.timer = self.create_timer(0.05, self.process_frame)

        # 4. HSV 閥值定義
        # 黃色線範圍 (主線)
        self.lower_yellow = np.array([20, 100, 100])
        self.upper_yellow = np.array([30, 255, 255])
        # 白色線範圍 (備用線)
        self.lower_white = np.array([0, 0, 180])
        self.upper_white = np.array([255, 50, 255])
        # 判斷質心的最小面積閾值 (避免雜訊干擾)
        self.area_threshold = 1000

        # 定義偏移量，依據線條所處位置調整 (單位: 像素)
        # 黃線在左側，所以道路中心 = 黃線中心 + offset
        self.offset_yellow = 300  
        # 白線在右側，所以道路中心 = 白線中心 - offset
        self.offset_white = 300

        self.get_logger().info("單線尋路節點已啟動。優先找黃線，找不到則切換白線；並依據偏移量計算道路中心。")

    def image_callback(self, msg):
        try:
            # ROS Image 消息轉成 OpenCV 格式
            frame = self.bridge.imgmsg_to_cv2(msg, desired_encoding='bgr8')
            self.last_frame = frame
            self.new_frame_available = True
        except Exception as e:
            self.get_logger().error("影像轉換錯誤: " + str(e))

    def process_frame(self):
        if not self.new_frame_available or self.last_frame is None:
            return

        # 取最新影像
        frame = self.last_frame.copy()
        self.new_frame_available = False

        # 將影像轉換到 HSV 色彩空間
        hsv = cv2.cvtColor(frame, cv2.COLOR_BGR2HSV)

        # --- 優先使用黃色遮罩 ---
        mask_yellow = cv2.inRange(hsv, self.lower_yellow, self.upper_yellow)
        cv2.imshow("Yellow Mask", mask_yellow)
        center = self.get_line_center(mask_yellow)
        detected_color = "yellow"

        # --- 如果黃色線無法有效偵測，則嘗試白色遮罩 ---
        if center is None:
            mask_white = cv2.inRange(hsv, self.lower_white, self.upper_white)
            cv2.imshow("White Mask", mask_white)
            center = self.get_line_center(mask_white)
            detected_color = "white"
        else:
            # 同時顯示白色遮罩方便調試
            mask_white = cv2.inRange(hsv, self.lower_white, self.upper_white)
            cv2.imshow("White Mask", mask_white)

        # 計算畫面中心 (假設相機正前方)
        h, w, _ = frame.shape
        frame_center = w // 2

        # 加入偏移調整，計算道路中心
        if center is not None:
            cx, cy = center
            if detected_color == "yellow":
                # 黃線在左，需將道路中心往右偏移一定距離
                road_center_x = cx + self.offset_yellow
            elif detected_color == "white":
                # 白線在右，需將道路中心往左偏移一定距離
                road_center_x = cx - self.offset_white
            else:
                road_center_x = cx  # 理論上不會發生
            
            # 將計算後的道路中心作為目標，畫出點供調試觀察
            cv2.circle(frame, (road_center_x, cy), 5, (0, 0, 255), -1)
            # 計算偏差：道路中心與影像中心的誤差
            error = road_center_x - frame_center
            cv2.putText(frame, f"Error: {error}", (10, 30), 
                        cv2.FONT_HERSHEY_SIMPLEX, 1, (0, 255, 0), 2)
            
            # 基本的比例控制 (P control)
            linear_speed = 0.05  # 固定前進速度
            angular_speed = -0.005 * error  # 根據偏差決定轉向，調參可根據實際情況進行修改
            
            twist = Twist()
            twist.linear.x = linear_speed
            twist.angular.z = angular_speed
            self.cmd_vel_pub.publish(twist)
            self.get_logger().info(
                f"檢測到 {detected_color} 線, 質心: ({cx},{cy}), 調整後道路中心: ({road_center_x},{cy})，error: {error}"
            )
        else:
            # 黃線和白線均無法偵測到，原地旋轉以搜尋
            twist = Twist()
            twist.linear.x = 0.0
            twist.angular.z = 0.3  # 可依據實際情況調整旋轉速度
            self.cmd_vel_pub.publish(twist)
            self.get_logger().info("未偵測到任何線，開始旋轉搜尋。")

        cv2.imshow("Line Tracking", frame)
        key = cv2.waitKey(1)
        if key == ord('q'):
            self.get_logger().info("使用者請求退出。")
            self.destroy_node()
            rclpy.shutdown()
            cv2.destroyAllWindows()

    def get_line_center(self, mask):
        """
        利用影像 moments 求遮罩內的質心。
        若遮罩面積 (m00) 大於設定閾值，回傳 (cx, cy)；否則回傳 None
        """
        M = cv2.moments(mask)
        if M["m00"] > self.area_threshold:
            cx = int(M["m10"] / M["m00"])
            cy = int(M["m01"] / M["m00"])
            return (cx, cy)
        else:
            return None

def main(args=None):
    rclpy.init(args=args)
    node = SingleLineTracker()
    try:
        rclpy.spin(node)
    except KeyboardInterrupt:
        node.get_logger().info("鍵盤中斷，節點關閉。")
    finally:
        node.destroy_node()
        rclpy.shutdown()
        cv2.destroyAllWindows()

if __name__ == '__main__':
    main()

# zhang ai



import rclpy
from rclpy.node import Node
from sensor_msgs.msg import Image, LaserScan
from geometry_msgs.msg import Twist
from cv_bridge import CvBridge

import cv2
import numpy as np
import math

class SingleLineWithSmartAvoid(Node):
    def __init__(self):
        super().__init__('single_line_with_smart_avoid')

        # 訂閱影像與雷達數據
        self.image_sub = self.create_subscription(
            Image, '/camera/image_raw', self.image_callback, 10)
        self.scan_sub = self.create_subscription(
            LaserScan, '/scan', self.scan_callback, 10)
        # 發布控制指令
        self.cmd_vel_pub = self.create_publisher(Twist, '/cmd_vel', 10)
        
        self.bridge = CvBridge()
        self.last_frame = None
        self.new_frame_available = False
        
        # 定時器，約20Hz
        self.timer = self.create_timer(0.05, self.process_frame)

        # HSV 參數設定
        self.lower_yellow = np.array([20, 100, 100])
        self.upper_yellow = np.array([30, 255, 255])
        self.lower_white = np.array([0, 0, 180])
        self.upper_white = np.array([255, 50, 255])
        self.area_threshold = 1000

        # 偏移量 (單位：像素)
        self.offset_yellow = 300   # 黃線在左 → 道路中心 = 黃線中心 + offset
        self.offset_white = 300    # 白線在右 → 道路中心 = 白線中心 - offset

        # 雷達參數
        self.center_distance = float('inf')
        self.left_distance = float('inf')
        self.right_distance = float('inf')
        self.obstacle_threshold = 0.5  # 障礙物觸發距離（公尺）
        self.clear_threshold = 0.7     # 障礙物清空標準

        # 模式：tracking（正常追線）或 avoiding（避障）
        self.mode = "tracking"
        self.avoid_direction = None  # "left" 或 "right"

        self.get_logger().info("微繞行閃避版節點啟動（融合雷達、視覺與 ROI），已加入白色過濾功能。")

    def image_callback(self, msg):
        try:
            frame = self.bridge.imgmsg_to_cv2(msg, desired_encoding='bgr8')
            self.last_frame = frame
            self.new_frame_available = True
        except Exception as e:
            self.get_logger().error("Image conversion error: " + str(e))

    def scan_callback(self, msg):
        """
        分析 LaserScan 資料：計算前方中心 (-15°～15°)、左側 (15°～75°) 與右側 (-75°～-15°) 區域內的最小距離。
        """
        self.center_distance = float('inf')
        self.left_distance = float('inf')
        self.right_distance = float('inf')
        for i, r in enumerate(msg.ranges):
            angle = msg.angle_min + i * msg.angle_increment
            if math.isnan(r) or r == float('inf'):
                continue
            if -math.radians(15) <= angle <= math.radians(15):
                if r < self.center_distance:
                    self.center_distance = r
            elif math.radians(15) < angle <= math.radians(75):
                if r < self.left_distance:
                    self.left_distance = r
            elif -math.radians(75) <= angle < -math.radians(15):
                if r < self.right_distance:
                    self.right_distance = r

    def get_line_center(self, mask):
        """
        利用 cv2.moments() 計算遮罩內的質心。
        若 m00 大於 area_threshold 則返回 (cx, cy)，否則返回 None。
        """
        M = cv2.moments(mask)
        if M["m00"] > self.area_threshold:
            cx = int(M["m10"] / M["m00"])
            cy = int(M["m01"] / M["m00"])
            return (cx, cy)
        else:
            return None

    def get_white_line_center(self, mask):
        """
        對白色遮罩進行形態學處理與輪廓分析，過濾障礙物等干擾，
        返回符合車道線特徵的白線質心。
        """
        # 形態學開運算去除小噪聲
        kernel = cv2.getStructuringElement(cv2.MORPH_RECT, (3, 3))
        mask_clean = cv2.morphologyEx(mask, cv2.MORPH_OPEN, kernel)
        # 找輪廓（僅保留外部輪廓）
        contours, _ = cv2.findContours(mask_clean, cv2.RETR_EXTERNAL, cv2.CHAIN_APPROX_SIMPLE)
        candidate_centers = []
        for cnt in contours:
            area = cv2.contourArea(cnt)
            if area < 200 or area > 2000:
                continue
            x, y, w_box, h_box = cv2.boundingRect(cnt)
            aspect_ratio = w_box / float(h_box)
            if aspect_ratio > 2.0:
                continue
            M = cv2.moments(cnt)
            if M["m00"] == 0:
                continue
            cx = int(M["m10"] / M["m00"])
            cy = int(M["m01"] / M["m00"])
            candidate_centers.append((cx, cy))
        if candidate_centers:
            return max(candidate_centers, key=lambda p: p[1])
        return None

    def process_frame(self):
        if not self.new_frame_available or self.last_frame is None:
            return

        # 取最新影像並重置標記
        frame = self.last_frame.copy()
        self.new_frame_available = False
        h, w, _ = frame.shape
        frame_center = w // 2

        # ── 加入 ROI：僅取畫面下部 40% (你可根據情況調整比例) ──
        roi_start = int(h * 0.6)
        roi = frame[roi_start:, :]
        
        # 如果要在 ROI 上顯示除錯，可顯示 ROI 畫面：
        cv2.imshow("ROI", roi)

        # 由於後續處理都在 ROI 區域進行，
        # 將 HSV 轉換於 ROI 區域中進行計算
        hsv = cv2.cvtColor(roi, cv2.COLOR_BGR2HSV)

        # 先依雷達數據判斷是否進入避障模式
        if self.mode == "tracking" and self.center_distance < self.obstacle_threshold:
            if self.left_distance < self.right_distance:
                self.avoid_direction = "right"  # 障礙在左側，向右偏移
            else:
                self.avoid_direction = "left"   # 障礙在右側，向左偏移
            self.mode = "avoiding"
            self.get_logger().info(f"障礙觸發！避障方向: {self.avoid_direction}")

        if self.mode == "avoiding":
            if self.center_distance > self.clear_threshold:
                self.mode = "tracking"
                self.avoid_direction = None
                self.get_logger().info("障礙物清除，恢復線追蹤模式。")
            else:
                twist = Twist()
                twist.linear.x = 0.04
                if self.avoid_direction == "right":
                    twist.angular.z = -0.3
                else:
                    twist.angular.z = 0.3
                self.cmd_vel_pub.publish(twist)
                cv2.imshow("Line Tracking", frame)
                cv2.waitKey(1)
                return

        # ── 正常線追蹤模式 ──
        # 優先嘗試黃色遮罩
        mask_yellow = cv2.inRange(hsv, self.lower_yellow, self.upper_yellow)
        cv2.imshow("Yellow Mask", mask_yellow)
        cv2.imshow("ROI", roi)
        center = self.get_line_center(mask_yellow)
        detected_color = "yellow"
        # 若黃色未找到，則嘗試白色遮罩並用過濾方法
        if center is None:
            mask_white = cv2.inRange(hsv, self.lower_white, self.upper_white)
            cv2.imshow("White Mask", mask_white)
            center = self.get_white_line_center(mask_white)
            detected_color = "white"
        else:
            mask_white = cv2.inRange(hsv, self.lower_white, self.upper_white)
            cv2.imshow("White Mask", mask_white)

        if center is not None:
            # center 為 ROI 相對座標，需補回 ROI 起始位置
            cx, cy = center
            cy_full = cy + roi_start

            if detected_color == "yellow":
                road_center_x = cx + self.offset_yellow
            elif detected_color == "white":
                road_center_x = cx - self.offset_white
            else:
                road_center_x = cx

            # 畫出道路中心（紅點），使用全圖座標
            cv2.circle(frame, (road_center_x, cy_full), 5, (0, 0, 255), -1)
            error = road_center_x - frame_center
            cv2.putText(frame, f"Error: {error}", (10, 30),
                        cv2.FONT_HERSHEY_SIMPLEX, 1, (0, 255, 0), 2)

            twist = Twist()
            twist.linear.x = 0.05
            twist.angular.z = -0.005 * error
            self.cmd_vel_pub.publish(twist)
            self.get_logger().info(f"追蹤 {detected_color} 線，質心: ({cx},{cy}), 調整後中心: ({road_center_x},{cy_full})，error: {error}")
        else:
            twist = Twist()
            twist.linear.x = 0.0
            twist.angular.z = 0.3
            self.cmd_vel_pub.publish(twist)
            self.get_logger().info("未偵測到線條，原地旋轉搜尋。")

        cv2.imshow("Line Tracking", frame)
        key = cv2.waitKey(1)
        if key == ord('q'):
            self.get_logger().info("使用者請求退出。")
            self.destroy_node()
            rclpy.shutdown()
            cv2.destroyAllWindows()

def main(args=None):
    rclpy.init(args=args)
    node = SingleLineWithSmartAvoid()
    try:
        rclpy.spin(node)
    except KeyboardInterrupt:
        node.get_logger().info("鍵盤中斷，節點關閉。")
    finally:
        node.destroy_node()
        rclpy.shutdown()
        cv2.destroyAllWindows()

if __name__ == '__main__':
    main()



# zhang ai initial

import rclpy
from rclpy.node import Node
from sensor_msgs.msg import Image, LaserScan
from geometry_msgs.msg import Twist
from cv_bridge import CvBridge

import cv2
import numpy as np
import math

class SingleLineWithSmartAvoid(Node):
    def __init__(self):
        super().__init__('single_line_with_smart_avoid')

        # 訂閱影像與雷達
        self.image_sub = self.create_subscription(
            Image, '/camera/image_raw', self.image_callback, 10)
        self.scan_sub = self.create_subscription(
            LaserScan, '/scan', self.scan_callback, 10)
        self.cmd_vel_pub = self.create_publisher(Twist, '/cmd_vel', 10)

        self.bridge = CvBridge()
        self.last_frame = None
        self.new_frame_available = False

        self.timer = self.create_timer(0.05, self.process_frame)

        # HSV 設定
        self.lower_yellow = np.array([20, 100, 100])
        self.upper_yellow = np.array([30, 255, 255])
        self.lower_white = np.array([0, 0, 180])
        self.upper_white = np.array([255, 50, 255])
        self.area_threshold = 1000

        # 偏移量設定
        self.offset_yellow = 300
        self.offset_white = 300

        # 雷達數據
        self.center_distance = float('inf')
        self.left_distance = float('inf')
        self.right_distance = float('inf')
        self.obstacle_threshold = 0.5  # 判斷有障礙物的距離
        self.clear_threshold = 0.7     # 判斷障礙物清空的距離

        # 運行模式
        self.mode = "tracking"  # tracking 或 avoiding
        self.avoid_direction = None  # "left" 或 "right"

        self.get_logger().info("微繞行閃避版節點啟動。")

    def image_callback(self, msg):
        try:
            frame = self.bridge.imgmsg_to_cv2(msg, desired_encoding='bgr8')
            self.last_frame = frame
            self.new_frame_available = True
        except Exception as e:
            self.get_logger().error("影像轉換錯誤: " + str(e))

    def scan_callback(self, msg):
        """
        分析雷達資料：中心、左側、右側的最小距離
        """
        self.center_distance = float('inf')
        self.left_distance = float('inf')
        self.right_distance = float('inf')

        for i, r in enumerate(msg.ranges):
            angle = msg.angle_min + i * msg.angle_increment
            if math.isnan(r) or r == float('inf'):
                continue
            if -math.radians(15) <= angle <= math.radians(15):
                if r < self.center_distance:
                    self.center_distance = r
            elif math.radians(15) < angle <= math.radians(75):
                if r < self.left_distance:
                    self.left_distance = r
            elif -math.radians(75) <= angle < -math.radians(15):
                if r < self.right_distance:
                    self.right_distance = r

    def process_frame(self):
        if not self.new_frame_available or self.last_frame is None:
            return

        frame = self.last_frame.copy()
        self.new_frame_available = False
        h, w, _ = frame.shape
        frame_center = w // 2

        # 🛑 判斷進入繞行模式
        if self.mode == "tracking" and self.center_distance < self.obstacle_threshold:
            if self.left_distance < self.right_distance:
                self.avoid_direction = "right"  # 左邊有障礙物，往右偏
            else:
                self.avoid_direction = "left"   # 右邊有障礙物，往左偏
            self.mode = "avoiding"
            self.get_logger().info(f"遇到障礙物！開始微繞行：{self.avoid_direction}")

        # ✅ 如果是 "avoiding" 模式
        if self.mode == "avoiding":
            if self.center_distance > self.clear_threshold:
                # 如果前方已經安全了，回到 tracking
                self.mode = "tracking"
                self.avoid_direction = None
                self.get_logger().info("障礙物清除，回到線追蹤模式")
            else:
                # 仍然在繞行中，持續微偏移前進
                twist = Twist()
                twist.linear.x = 0.04
                if self.avoid_direction == "right":
                    twist.angular.z = -0.3
                else:
                    twist.angular.z = 0.3
                self.cmd_vel_pub.publish(twist)

                cv2.imshow("Line Tracking", frame)
                cv2.waitKey(1)
                return

        # ✅ 正常線追蹤模式
        hsv = cv2.cvtColor(frame, cv2.COLOR_BGR2HSV)
        mask_yellow = cv2.inRange(hsv, self.lower_yellow, self.upper_yellow)
        cv2.imshow("Yellow Mask", mask_yellow)
        center = self.get_line_center(mask_yellow)
        detected_color = "yellow"

        if center is None:
            mask_white = cv2.inRange(hsv, self.lower_white, self.upper_white)
            cv2.imshow("White Mask", mask_white)
            center = self.get_line_center(mask_white)
            detected_color = "white"
        else:
            mask_white = cv2.inRange(hsv, self.lower_white, self.upper_white)
            cv2.imshow("White Mask", mask_white)

        if center is not None:
            cx, cy = center
            if detected_color == "yellow":
                road_center_x = cx + self.offset_yellow
            else:
                road_center_x = cx - self.offset_white

            cv2.circle(frame, (road_center_x, cy), 5, (0, 0, 255), -1)
            error = road_center_x - frame_center
            cv2.putText(frame, f"Error: {error}", (10, 30),
                        cv2.FONT_HERSHEY_SIMPLEX, 1, (0, 255, 0), 2)

            twist = Twist()
            twist.linear.x = 0.05
            twist.angular.z = -0.005 * error
            self.cmd_vel_pub.publish(twist)
        else:
            twist = Twist()
            twist.linear.x = 0.0
            twist.angular.z = 0.3
            self.cmd_vel_pub.publish(twist)
            self.get_logger().info("未偵測到線條，原地旋轉搜尋。")

        cv2.imshow("Line Tracking", frame)
        key = cv2.waitKey(1)
        if key == ord('q'):
            self.get_logger().info("使用者請求退出。")
            self.destroy_node()
            rclpy.shutdown()
            cv2.destroyAllWindows()

    def get_line_center(self, mask):
        """
        利用影像 moments 計算遮罩內的質心。
        """
        M = cv2.moments(mask)
        if M["m00"] > self.area_threshold:
            cx = int(M["m10"] / M["m00"])
            cy = int(M["m01"] / M["m00"])
            return (cx, cy)
        else:
            return None

def main(args=None):
    rclpy.init(args=args)
    node = SingleLineWithSmartAvoid()
    try:
        rclpy.spin(node)
    except KeyboardInterrupt:
        node.get_logger().info("鍵盤中斷，節點關閉。")
    finally:
        node.destroy_node()
        rclpy.shutdown()
        cv2.destroyAllWindows()

if __name__ == '__main__':
    main()
